# MiniPlaces Scene Recognition - Comprehensive Experiments

## BMI/CS 771 Assignment 2

This notebook implements systematic experiments to improve upon the default SimpleNet configuration.
Based on the lecture slides covering advanced neural network training techniques.

### Key Areas of Experimentation:
1. **Architecture Modifications** - More layers, BatchNorm, skip connections, parallel branches
2. **Optimizer Selection** - SGD+momentum vs Adam vs AdamW
3. **Learning Rate Schedules** - Cosine annealing, step decay, warmup strategies
4. **Regularization Techniques** - Dropout, Stochastic Depth, weight decay
5. **Data Augmentation** - Color jitter, rotation, crop strategies

## Part 1: Setup and Imports

In [ ]:
import os
import sys
import yaml
import copy
import subprocess
from datetime import datetime

# Add the code directory to path
CODE_DIR = os.path.dirname(os.path.abspath('.'))
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Part 2: Understanding the Baseline (SimpleNet)

### Current Architecture Analysis

The default SimpleNet has this structure:
- **Conv1**: 7x7 conv, stride 2 (3 → 64 channels) + MaxPool → 32x32
- **Conv2 Bottleneck**: 1x1(64→64) → 3x3(64→64) → 1x1(64→256) + MaxPool → 16x16  
- **Conv3 Bottleneck**: 1x1(256→128) → 3x3(128→128) → 1x1(128→512) → 16x16
- **Global Average Pool** → FC(512→100)

### Issues with Current Design:
1. **No Batch Normalization** - Training can be unstable, slower convergence
2. **No Skip Connections** - Gradient flow issues in deeper networks
3. **No Dropout/Regularization** - Potential overfitting
4. **Limited Depth** - Only 2 bottleneck blocks

In [ ]:
# Baseline configuration - this is what we're improving upon
BASELINE_CONFIG = {
    "output_folder": "../logs",
    "model_type": "CNN",
    "dataset": {
        "root_folder": "../data"
    },
    "loader": {
        "batch_size": 512,
        "num_workers": 4
    },
    "input": {
        "img_min_size": 144,
        "img_crop_size": 128,
        "img_color_jitter": 0.15,
        "img_rotation": 15,
        "img_mean": [0.485, 0.456, 0.406],
        "img_std": [0.229, 0.224, 0.225]
    },
    "model": {
        "num_classes": 100
    },
    "opt": {
        "type": "SGD",
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "learning_rate": 0.1,
        "epochs": 60,
        "warmup": True,
        "warmup_epochs": 1,
        "schedule_type": "cosine",
        "schedule_steps": [],
        "schedule_gamma": 0.1
    }
}

print("Baseline Configuration:")
print(yaml.dump(BASELINE_CONFIG, default_flow_style=False))

---

# SECTION A: ARCHITECTURE EXPERIMENTS

## Based on Lecture Slides:
- **He Initialization**: For ReLU networks, use `fan_out` mode with gain=√2
- **Batch Normalization**: Normalize activations → faster training, acts as regularizer
- **Skip Connections (ResNet)**: Enable training of deeper networks via gradient highways
- **Parallel Branches (Inception)**: Multi-scale feature extraction

---

## Experiment A1: Adding Batch Normalization

### Theory (from lecture):
- BatchNorm normalizes activations to zero mean, unit variance
- Adds learnable scale (γ) and shift (β) parameters
- **Benefits**:
  - Allows higher learning rates → faster convergence
  - Acts as implicit regularization
  - Reduces internal covariate shift
- **Placement**: After Conv, BEFORE ReLU (standard practice)

In [ ]:
# Architecture A1: SimpleNet with Batch Normalization
# This code should be added to cls_libs/modeling.py

SIMPLENET_BN_CODE = '''
class SimpleNetBN(nn.Module):
    """
    SimpleNet with Batch Normalization added after each Conv layer.
    
    EXPERIMENT A1: Testing the impact of BatchNorm on:
    - Training stability (can use higher learning rates)
    - Convergence speed (fewer epochs to reach same accuracy)
    - Final accuracy (regularization effect)
    
    Architecture: Conv-BN-ReLU pattern throughout
    """
    def __init__(self, conv_op=nn.Conv2d, num_classes=100):
        super(SimpleNetBN, self).__init__()
        
        self.features = nn.Sequential(
            # conv1 block: 7x7 conv with BatchNorm
            # Input: 3x128x128 → Output: 64x64x64 (after stride 2)
            conv_op(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),  # BatchNorm BEFORE ReLU
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),  # → 64x32x32
            
            # conv2 block: bottleneck with BatchNorm
            # Bottleneck: 1x1 reduce → 3x3 process → 1x1 expand
            conv_op(64, 64, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            conv_op(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            conv_op(64, 256, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),  # → 256x16x16
            
            # conv3 block: bottleneck with BatchNorm
            conv_op(256, 128, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            conv_op(128, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            conv_op(128, 512, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # He initialization for ReLU (from lecture)
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
'''

print("Experiment A1: SimpleNet with BatchNorm")
print("="*50)
print(SIMPLENET_BN_CODE)

## Experiment A2: Adding Skip Connections (ResNet-style)

### Theory (from lecture):
- **Problem**: Deep networks suffer from vanishing gradients
- **Solution**: Skip connections create "gradient highways"
- **Formula**: `output = F(x) + x` (identity shortcut)
- If dimensions don't match, use 1x1 conv to project
- **Benefits**:
  - Can train much deeper networks (100+ layers)
  - Easier optimization landscape
  - Better gradient flow during backprop

In [ ]:
# Architecture A2: ResNet-style with Skip Connections

RESNET_STYLE_CODE = '''
class ResidualBlock(nn.Module):
    """
    Basic Residual Block with skip connection.
    
    EXPERIMENT A2: Tests gradient flow improvement via skip connections.
    
    The skip connection allows gradients to flow directly backward,
    bypassing the nonlinearities. This enables training of deeper networks.
    
    output = F(x) + shortcut(x)
    where F = Conv-BN-ReLU-Conv-BN
    and shortcut = identity if dims match, else 1x1 conv
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        
        # Main path: two 3x3 convolutions
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut connection (identity or projection)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            # Need to match dimensions with 1x1 conv
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # Skip connection: add input to output
        out += self.shortcut(identity)
        out = self.relu(out)  # ReLU after addition
        
        return out


class SimpleResNet(nn.Module):
    """
    ResNet-style architecture for MiniPlaces.
    
    EXPERIMENT A2: 4-stage ResNet with skip connections
    
    Stage 1: stem (7x7 conv + pool) → 64 channels
    Stage 2: 2 residual blocks → 64 channels
    Stage 3: 2 residual blocks → 128 channels (stride 2)
    Stage 4: 2 residual blocks → 256 channels (stride 2)
    Stage 5: 2 residual blocks → 512 channels (stride 2)
    
    Total: 8 residual blocks = 16 conv layers + stem = ~18 layers
    """
    def __init__(self, num_classes=100):
        super(SimpleResNet, self).__init__()
        
        # Stem: initial conv + pool
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )  # 128 → 32
        
        # Residual stages
        self.layer1 = self._make_layer(64, 64, num_blocks=2, stride=1)
        self.layer2 = self._make_layer(64, 128, num_blocks=2, stride=2)   # 32 → 16
        self.layer3 = self._make_layer(128, 256, num_blocks=2, stride=2)  # 16 → 8
        self.layer4 = self._make_layer(256, 512, num_blocks=2, stride=2)  # 8 → 4
        
        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        
        # Initialize weights
        self._initialize_weights()
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        # First block may have stride > 1 for downsampling
        layers.append(ResidualBlock(in_channels, out_channels, stride))
        # Remaining blocks have stride 1
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
'''

print("Experiment A2: ResNet-style with Skip Connections")
print("="*50)
print(RESNET_STYLE_CODE)

## Experiment A3: Parallel Branches (Inception-style)

### Theory (from lecture):
- **Multi-scale feature extraction**: Different kernel sizes capture different patterns
- 1x1: point-wise features, channel mixing
- 3x3: local spatial patterns
- 5x5: larger spatial patterns
- **Dimension reduction**: 1x1 convs reduce channels before expensive 3x3/5x5
- **Benefits**:
  - Captures features at multiple scales simultaneously
  - Efficient due to dimensionality reduction

In [ ]:
# Architecture A3: Inception-style Parallel Branches

INCEPTION_STYLE_CODE = '''
class InceptionBlock(nn.Module):
    """
    Inception-style block with parallel branches.
    
    EXPERIMENT A3: Tests multi-scale feature extraction.
    
    Four parallel branches:
    1. 1x1 conv: point-wise features (32 channels)
    2. 1x1 → 3x3 conv: local patterns (64 channels)
    3. 1x1 → 5x5 conv: larger patterns (32 channels)
    4. 3x3 pool → 1x1 conv: pooled features (32 channels)
    
    Total output: 32 + 64 + 32 + 32 = 160 channels (concatenated)
    """
    def __init__(self, in_channels, ch1x1, ch3x3_reduce, ch3x3, 
                 ch5x5_reduce, ch5x5, pool_proj):
        super(InceptionBlock, self).__init__()
        
        # Branch 1: 1x1 convolution
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch1x1, kernel_size=1, bias=False),
            nn.BatchNorm2d(ch1x1),
            nn.ReLU(inplace=True)
        )
        
        # Branch 2: 1x1 reduce → 3x3
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3_reduce, kernel_size=1, bias=False),
            nn.BatchNorm2d(ch3x3_reduce),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch3x3_reduce, ch3x3, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(ch3x3),
            nn.ReLU(inplace=True)
        )
        
        # Branch 3: 1x1 reduce → 5x5 (implemented as 3x3 + 3x3 for efficiency)
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5_reduce, kernel_size=1, bias=False),
            nn.BatchNorm2d(ch5x5_reduce),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch5x5_reduce, ch5x5, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(ch5x5),
            nn.ReLU(inplace=True)
        )
        
        # Branch 4: Pool → 1x1
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1, bias=False),
            nn.BatchNorm2d(pool_proj),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Run all branches in parallel
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        
        # Concatenate along channel dimension
        return torch.cat([b1, b2, b3, b4], dim=1)


class SimpleInceptionNet(nn.Module):
    """
    Inception-style network for MiniPlaces.
    
    EXPERIMENT A3: Multi-scale feature extraction
    
    Architecture:
    - Stem: 7x7 conv + pool → 64 channels
    - Inception Block 1: 64 → 256 channels
    - Inception Block 2: 256 → 512 channels
    - GAP + FC
    """
    def __init__(self, num_classes=100):
        super(SimpleInceptionNet, self).__init__()
        
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )  # 128 → 32
        
        # Inception blocks
        # InceptionBlock(in, 1x1, 3x3_red, 3x3, 5x5_red, 5x5, pool_proj)
        self.inception1 = InceptionBlock(64, 64, 64, 96, 16, 32, 32)  # out: 64+96+32+32=224
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)  # 32 → 16
        
        self.inception2 = InceptionBlock(224, 128, 96, 128, 32, 64, 64)  # out: 128+128+64+64=384
        self.pool2 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)  # 16 → 8
        
        self.inception3 = InceptionBlock(384, 160, 128, 192, 48, 96, 64)  # out: 160+192+96+64=512
        
        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)  # Inception uses dropout
        self.fc = nn.Linear(512, num_classes)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.inception1(x)
        x = self.pool1(x)
        x = self.inception2(x)
        x = self.pool2(x)
        x = self.inception3(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x
'''

print("Experiment A3: Inception-style Parallel Branches")
print("="*50)
print(INCEPTION_STYLE_CODE)

## Experiment A4: Deeper Network with Stochastic Depth

### Theory (from lecture):
- **Stochastic Depth (DropPath)**: Randomly drop entire residual blocks during training
- **Formula**: `output = x + drop_path(F(x))` where drop_path zeros out F(x) with probability p
- **Linear decay**: Drop probability increases with depth (deeper = more likely to drop)
- **Benefits**:
  - Regularization effect (prevents overfitting)
  - Trains an implicit ensemble of networks
  - Enables training of very deep networks (100+ layers)

In [ ]:
# Architecture A4: Deeper Network with Stochastic Depth

DEEP_STOCHASTIC_CODE = '''
# DropPath is already imported from blocks.py
# from .blocks import DropPath

class ResidualBlockSD(nn.Module):
    """
    Residual Block with Stochastic Depth (DropPath).
    
    EXPERIMENT A4: Tests regularization via stochastic depth.
    
    During training, the residual path is randomly dropped with probability p.
    During inference, outputs are scaled by (1-p).
    
    This acts as regularization and trains an implicit ensemble.
    """
    def __init__(self, in_channels, out_channels, stride=1, drop_path_rate=0.0):
        super(ResidualBlockSD, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Stochastic Depth (DropPath)
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0 else nn.Identity()
        
        # Shortcut
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # Apply stochastic depth to residual path
        out = self.drop_path(out)
        
        out = identity + out
        out = self.relu(out)
        return out


class DeepResNetSD(nn.Module):
    """
    Deep ResNet with Stochastic Depth for MiniPlaces.
    
    EXPERIMENT A4: Deeper architecture (26 layers) with regularization
    
    Uses linear decay for drop path rate:
    - First block: 0% drop rate
    - Last block: max_drop_rate (e.g., 0.2)
    
    This enables training deeper networks while preventing overfitting.
    """
    def __init__(self, num_classes=100, drop_path_rate=0.2):
        super(DeepResNetSD, self).__init__()
        
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        
        # Calculate drop path rates with linear decay
        # Total blocks: 3 + 4 + 4 + 3 = 14 blocks
        total_blocks = 14
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, total_blocks)]
        
        # Build layers with increasing drop rates
        self.layer1 = self._make_layer(64, 64, 3, stride=1, drop_rates=dpr[0:3])
        self.layer2 = self._make_layer(64, 128, 4, stride=2, drop_rates=dpr[3:7])
        self.layer3 = self._make_layer(128, 256, 4, stride=2, drop_rates=dpr[7:11])
        self.layer4 = self._make_layer(256, 512, 3, stride=2, drop_rates=dpr[11:14])
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        
        self._initialize_weights()
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride, drop_rates):
        layers = []
        layers.append(ResidualBlockSD(in_channels, out_channels, stride, drop_rates[0]))
        for i in range(1, num_blocks):
            layers.append(ResidualBlockSD(out_channels, out_channels, 1, drop_rates[i]))
        return nn.Sequential(*layers)
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
'''

print("Experiment A4: Deep Network with Stochastic Depth")
print("="*50)
print(DEEP_STOCHASTIC_CODE)

---

# SECTION B: OPTIMIZER EXPERIMENTS

## Based on Lecture Slides:
- **SGD + Momentum**: Standard, well-understood, often best for final accuracy
- **Adam**: Adaptive learning rates per parameter, faster initial convergence
- **AdamW**: Decoupled weight decay, better generalization than Adam

---

In [ ]:
# Experiment B1: SGD with Momentum (Baseline)
# This is the current default - included for comparison

CONFIG_SGD_MOMENTUM = {
    "output_folder": "../logs",
    "model_type": "CNN",
    "dataset": {"root_folder": "../data"},
    "opt": {
        "type": "SGD",
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "learning_rate": 0.1,
        "epochs": 60,
        "warmup": True,
        "warmup_epochs": 1,
        "schedule_type": "cosine"
    }
}

print("Experiment B1: SGD with Momentum")
print("="*50)
print("""
THEORY (from lecture):
- Momentum helps escape local minima and smooths gradients
- v_t = β * v_{t-1} + g_t  (velocity accumulation)
- θ_t = θ_{t-1} - lr * v_t
- β = 0.9 means ~10 past gradients contribute significantly

EXPECTED BEHAVIOR:
- Slower initial convergence than Adam
- Often achieves best final accuracy
- Sensitive to learning rate choice

HYPERPARAMETERS:
- learning_rate: 0.1 (high, requires warmup)
- momentum: 0.9 (standard)
- weight_decay: 1e-4 (L2 regularization)
""")
print(yaml.dump(CONFIG_SGD_MOMENTUM['opt'], default_flow_style=False))

In [ ]:
# Experiment B2: Adam Optimizer

CONFIG_ADAM = {
    "output_folder": "../logs",
    "model_type": "CNN",
    "dataset": {"root_folder": "../data"},
    "opt": {
        "type": "Adam",
        "weight_decay": 1e-4,
        "learning_rate": 0.001,  # Adam needs lower LR
        "epochs": 60,
        "warmup": False,  # Adam doesn't need warmup
        "warmup_epochs": 0,
        "schedule_type": "cosine"
    }
}

print("Experiment B2: Adam Optimizer")
print("="*50)
print("""
THEORY (from lecture):
- Combines momentum with adaptive per-parameter learning rates
- m_t = β1 * m_{t-1} + (1-β1) * g_t  (first moment)
- v_t = β2 * v_{t-1} + (1-β2) * g_t²  (second moment)
- θ_t = θ_{t-1} - lr * m̂_t / (√v̂_t + ε)

EXPECTED BEHAVIOR:
- Faster initial convergence (good for quick experiments)
- May have slightly worse generalization than SGD
- Less sensitive to learning rate

HYPERPARAMETERS:
- learning_rate: 0.001 (typical for Adam)
- betas: (0.9, 0.999) (default)
- weight_decay: 1e-4 (L2 regularization)

NOTE: The train_cls.py would need modification to support Adam.
""")
print(yaml.dump(CONFIG_ADAM['opt'], default_flow_style=False))

In [ ]:
# Experiment B3: AdamW (Decoupled Weight Decay)

CONFIG_ADAMW = {
    "output_folder": "../logs",
    "model_type": "CNN",
    "dataset": {"root_folder": "../data"},
    "opt": {
        "type": "AdamW",
        "weight_decay": 0.05,  # AdamW uses higher weight decay
        "learning_rate": 0.001,
        "epochs": 60,
        "warmup": True,
        "warmup_epochs": 5,  # Longer warmup helps AdamW
        "schedule_type": "cosine"
    }
}

print("Experiment B3: AdamW (Decoupled Weight Decay)")
print("="*50)
print("""
THEORY (from lecture):
- Fixes weight decay in Adam by decoupling it from gradient updates
- Standard Adam: θ = θ - lr * (m̂/(√v̂+ε) + λθ)  [L2 affected by adaptive LR]
- AdamW: θ = θ - lr * m̂/(√v̂+ε) - lr * λθ  [true weight decay]

EXPECTED BEHAVIOR:
- Better generalization than Adam (closer to SGD)
- Recommended for transformers and modern architectures
- Benefits from longer warmup

HYPERPARAMETERS:
- learning_rate: 0.001
- weight_decay: 0.05 (higher than L2 in SGD/Adam)
- warmup_epochs: 5 (longer warmup)
""")
print(yaml.dump(CONFIG_ADAMW['opt'], default_flow_style=False))

---

# SECTION C: LEARNING RATE SCHEDULE EXPERIMENTS

## Based on Lecture Slides:
- **Cosine Annealing**: Smooth decay, currently best practice
- **Step Decay**: Simple, discrete drops
- **Warmup**: Prevents early instability with high learning rates

---

In [ ]:
# Experiment C1: Cosine Annealing (Baseline)

CONFIG_COSINE = {
    "opt": {
        "type": "SGD",
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "learning_rate": 0.1,
        "epochs": 60,
        "warmup": True,
        "warmup_epochs": 1,
        "schedule_type": "cosine",  # Cosine annealing
        "schedule_steps": [],
        "schedule_gamma": 0.1
    }
}

print("Experiment C1: Cosine Annealing Schedule")
print("="*50)
print("""
THEORY (from lecture):
- lr(t) = η_min + 0.5 * (η_max - η_min) * (1 + cos(π * t / T))
- Smooth decay from max to min learning rate
- No abrupt transitions (unlike step decay)

EXPECTED BEHAVIOR:
- Current best practice for most architectures
- Smooth training dynamics
- Good final accuracy

VISUALIZATION:
  LR |
0.1 |****
    |    ****
    |        ***
    |           ***
    |              **
    |                **
~0  +-------------------> Epoch
    0        30        60
""")

In [ ]:
# Experiment C2: Step Decay

CONFIG_STEP = {
    "opt": {
        "type": "SGD",
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "learning_rate": 0.1,
        "epochs": 60,
        "warmup": True,
        "warmup_epochs": 1,
        "schedule_type": "multistep",  # Step decay
        "schedule_steps": [20, 40],  # Drop at epochs 20 and 40
        "schedule_gamma": 0.1  # Multiply LR by 0.1 at each step
    }
}

print("Experiment C2: Step Decay Schedule")
print("="*50)
print("""
THEORY (from lecture):
- lr(t) = lr_0 * γ^(step_count)
- Discrete drops at specified epochs
- Classic approach (used in VGG, ResNet papers)

EXPECTED BEHAVIOR:
- Sudden accuracy jumps after LR drops
- Training loss may plateau before drops
- Simple to tune (just choose drop epochs)

VISUALIZATION:
  LR |
0.1 |*********
    |         |
0.01|         *********
    |                   |
0.001                   **********
    +-------------------> Epoch
    0    20    40    60

Steps: epoch 20, 40 (multiply by 0.1)
""")

In [ ]:
# Experiment C3: Extended Warmup

CONFIG_LONG_WARMUP = {
    "opt": {
        "type": "SGD",
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "learning_rate": 0.2,  # Can use higher LR with longer warmup
        "epochs": 60,
        "warmup": True,
        "warmup_epochs": 5,  # 5 epochs of linear warmup
        "schedule_type": "cosine"
    }
}

print("Experiment C3: Extended Warmup + Higher LR")
print("="*50)
print("""
THEORY (from lecture):
- Warmup: Gradually increase LR from 0 to target
- Prevents early training instability
- Longer warmup allows higher peak learning rate

EXPECTED BEHAVIOR:
- Smoother early training
- Can use higher learning rates safely
- Important for large batch sizes

VISUALIZATION:
  LR |
0.2 |      *****
    |    **     ***
    |  **           **
    |**               **
~0  +-------------------> Epoch
    0  5      35     60
    |--| warmup
""")

---

# SECTION D: REGULARIZATION EXPERIMENTS

## Based on Lecture Slides:
- **Weight Decay (L2)**: Penalize large weights, improve generalization
- **Dropout**: Random neuron dropping, prevents co-adaptation
- **Stochastic Depth**: Drop entire layers/blocks
- **Label Smoothing**: Prevents overconfident predictions

---

In [ ]:
# Experiment D1: Stronger Weight Decay

CONFIG_HIGH_WD = {
    "opt": {
        "type": "SGD",
        "momentum": 0.9,
        "weight_decay": 5e-4,  # 5x stronger than default
        "learning_rate": 0.1,
        "epochs": 60,
        "warmup": True,
        "warmup_epochs": 1,
        "schedule_type": "cosine"
    }
}

print("Experiment D1: Stronger Weight Decay")
print("="*50)
print("""
THEORY (from lecture):
- L2 regularization: Loss_total = Loss_task + λ||θ||²
- Equivalent to adding λθ to gradients (for SGD)
- Pushes weights toward zero → simpler models

EXPECTED BEHAVIOR:
- Reduced overfitting
- Lower training accuracy, potentially higher val accuracy
- May need adjustment for different architectures

HYPERPARAMETERS:
- weight_decay: 5e-4 (vs default 1e-4)
- Try: 1e-5, 1e-4, 5e-4, 1e-3
""")

In [ ]:
# Experiment D2: Adding Dropout
# This requires architecture modification

DROPOUT_CODE = '''
class SimpleNetDropout(nn.Module):
    """
    SimpleNet with Dropout regularization.
    
    EXPERIMENT D2: Tests dropout for preventing overfitting.
    
    Dropout placement:
    - After conv blocks (spatial dropout for CNNs)
    - Before final FC layer (standard for classification)
    
    During training: randomly zero out neurons with probability p
    During inference: scale activations by (1-p)
    """
    def __init__(self, conv_op=nn.Conv2d, num_classes=100, dropout_rate=0.5):
        super(SimpleNetDropout, self).__init__()
        
        self.features = nn.Sequential(
            conv_op(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            
            # Bottleneck 1
            conv_op(64, 64, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            conv_op(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            conv_op(64, 256, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate * 0.5),  # Spatial dropout (lower rate for early layers)
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            
            # Bottleneck 2
            conv_op(256, 128, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            conv_op(128, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            conv_op(128, 512, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate * 0.5),  # Spatial dropout
        )
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout_rate)  # Standard dropout before FC
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)  # Dropout before classifier
        x = self.fc(x)
        return x
'''

print("Experiment D2: Dropout Regularization")
print("="*50)
print("""
THEORY (from lecture):
- Randomly zero out activations during training
- Prevents co-adaptation of neurons
- Trains an implicit ensemble of sub-networks

EXPECTED BEHAVIOR:
- Lower training accuracy
- Better generalization (higher val accuracy)
- Works best with larger models

HYPERPARAMETERS:
- dropout_rate: 0.5 (standard for FC layers)
- spatial_dropout_rate: 0.25 (lower for conv layers)
""")
print(DROPOUT_CODE)

In [ ]:
# Experiment D3: Label Smoothing
# Requires modification to training script

LABEL_SMOOTHING_CODE = '''
class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross-Entropy with Label Smoothing.
    
    EXPERIMENT D3: Tests label smoothing for better generalization.
    
    Instead of hard labels (1 for correct class, 0 for others):
    - Correct class: (1 - ε) + ε/K
    - Other classes: ε/K
    
    where ε is smoothing factor and K is number of classes.
    
    This prevents overconfident predictions and improves calibration.
    """
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing
    
    def forward(self, x, target):
        # x: (N, C) logits
        # target: (N,) class indices
        
        logprobs = torch.nn.functional.log_softmax(x, dim=-1)
        
        # NLL for correct class
        nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1))
        nll_loss = nll_loss.squeeze(1)
        
        # Smooth loss: mean of all log probs
        smooth_loss = -logprobs.mean(dim=-1)
        
        # Combine
        loss = self.confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()


# Usage in train_cls.py:
# criterion = LabelSmoothingCrossEntropy(smoothing=0.1).to(device)
# instead of:
# criterion = torch.nn.CrossEntropyLoss().to(device)
'''

print("Experiment D3: Label Smoothing")
print("="*50)
print("""
THEORY (from lecture):
- Hard labels: [0, 0, 1, 0, ...] → model becomes overconfident
- Soft labels: [0.001, 0.001, 0.9, 0.001, ...] → better calibration
- Prevents model from being too certain about any prediction

EXPECTED BEHAVIOR:
- Slightly lower training accuracy
- Better calibrated confidence scores
- Improved generalization

HYPERPARAMETERS:
- smoothing: 0.1 (typical value)
- Try: 0.05, 0.1, 0.2
""")
print(LABEL_SMOOTHING_CODE)

---

# SECTION E: DATA AUGMENTATION EXPERIMENTS

## Based on Lecture Slides:
- **Geometric**: Flip, crop, rotate, scale
- **Photometric**: Color jitter, brightness, contrast
- **Advanced**: Mixup, CutOut, AutoAugment

---

In [ ]:
# Experiment E1: Stronger Data Augmentation

CONFIG_STRONG_AUG = {
    "input": {
        "img_min_size": 160,  # Larger initial size for more crop variety
        "img_crop_size": 128,
        "img_color_jitter": 0.3,  # 2x stronger color jitter (default: 0.15)
        "img_rotation": 25,  # Stronger rotation (default: 15)
        "img_mean": [0.485, 0.456, 0.406],
        "img_std": [0.229, 0.224, 0.225]
    }
}

print("Experiment E1: Stronger Data Augmentation")
print("="*50)
print("""
THEORY (from lecture):
- Data augmentation expands effective training set
- Creates more diverse training samples
- Forces model to learn invariant features

CHANGES FROM DEFAULT:
- img_min_size: 160 (was 144) - more crop variety
- img_color_jitter: 0.3 (was 0.15) - stronger color perturbation  
- img_rotation: 25° (was 15°) - more rotation variety

EXPECTED BEHAVIOR:
- Lower training accuracy (harder task)
- Better validation accuracy (better generalization)
- May need longer training
""")

In [ ]:
# Experiment E2: Implementing Mixup
# Requires modification to training loop

MIXUP_CODE = '''
import numpy as np

def mixup_data(x, y, alpha=0.2):
    """
    Mixup: Create convex combinations of training pairs.
    
    EXPERIMENT E2: Tests mixup augmentation.
    
    For each batch:
    1. Sample λ ~ Beta(α, α)
    2. Shuffle batch to get pairs (x_i, y_i), (x_j, y_j)
    3. Create mixed samples:
       x_mixed = λ * x_i + (1 - λ) * x_j
       y_mixed = λ * y_i + (1 - λ) * y_j
    
    Args:
        x: Input images (N, C, H, W)
        y: Labels (N,)
        alpha: Mixup strength (higher = more mixing)
    
    Returns:
        mixed_x, y_a, y_b, lam
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """
    Compute mixup loss as weighted sum of two losses.
    """
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# Usage in training loop:
# for images, labels in train_loader:
#     images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.2)
#     outputs = model(images)
#     loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
'''

print("Experiment E2: Mixup Augmentation")
print("="*50)
print("""
THEORY (from lecture):
- Mixup: linear interpolation between training examples
- x_new = λ * x_i + (1-λ) * x_j
- y_new = λ * y_i + (1-λ) * y_j
- Creates "soft" training targets

EXPECTED BEHAVIOR:
- Smoother decision boundaries
- Better generalization
- Reduces overconfidence

HYPERPARAMETERS:
- alpha: 0.2 (typical for image classification)
- Try: 0.1, 0.2, 0.4
""")
print(MIXUP_CODE)

---

# SECTION F: RUNNING EXPERIMENTS

## Helper Functions and Commands

---

In [ ]:
def create_config_file(config_dict, filename):
    """
    Create a YAML config file for training.
    Merges with baseline config to ensure all required fields exist.
    """
    full_config = copy.deepcopy(BASELINE_CONFIG)
    
    # Deep merge
    def merge(base, override):
        for key, value in override.items():
            if key in base and isinstance(base[key], dict) and isinstance(value, dict):
                merge(base[key], value)
            else:
                base[key] = value
    
    merge(full_config, config_dict)
    
    filepath = f"../configs/{filename}"
    with open(filepath, 'w') as f:
        yaml.dump(full_config, f, default_flow_style=False)
    
    print(f"Created config: {filepath}")
    return filepath


def run_experiment(config_file, experiment_name, print_freq=50, ckpt_freq=5):
    """
    Run a training experiment with the specified config.
    """
    cmd = f"python train_cls.py {config_file} --output {experiment_name} --print-freq {print_freq} --ckpt-freq {ckpt_freq}"
    print(f"\n{'='*60}")
    print(f"Running Experiment: {experiment_name}")
    print(f"Command: {cmd}")
    print(f"{'='*60}\n")
    
    # Uncomment to actually run:
    # subprocess.run(cmd.split())
    return cmd

In [ ]:
# Create all experiment configurations

experiments = [
    # Baseline
    ({}, "exp_baseline"),
    
    # Optimizer experiments
    ({"opt": {"weight_decay": 5e-4}}, "exp_wd_5e4"),
    ({"opt": {"learning_rate": 0.2, "warmup_epochs": 5}}, "exp_lr02_warmup5"),
    ({"opt": {"schedule_type": "multistep", "schedule_steps": [20, 40]}}, "exp_step_decay"),
    
    # Data augmentation experiments
    ({"input": {"img_color_jitter": 0.3, "img_rotation": 25}}, "exp_strong_aug"),
    ({"input": {"img_min_size": 160}}, "exp_larger_crop"),
    
    # Combined experiments (architecture changes require code modifications)
    ({"opt": {"weight_decay": 5e-4, "learning_rate": 0.15, "warmup_epochs": 3},
      "input": {"img_color_jitter": 0.25}}, "exp_combined_v1"),
]

print("Experiment Configurations:")
print("="*60)
for config, name in experiments:
    print(f"\n{name}:")
    if config:
        print(yaml.dump(config, default_flow_style=False))
    else:
        print("  (Baseline - no changes)")

In [ ]:
# Commands to run experiments

print("="*60)
print("TRAINING COMMANDS")
print("="*60)
print("""
Navigate to the code directory and run:

cd /path/to/assignment2_release/code

# Baseline experiment
python train_cls.py ./configs/cls_miniplaces_cnn.yaml --print-freq 50 --ckpt-freq 5 --output baseline

# After modifying modeling.py with new architectures:
# (Update default_cnn_model = YourNewModel at the bottom of modeling.py)

# Experiment A1: BatchNorm
python train_cls.py ./configs/cls_miniplaces_cnn.yaml --print-freq 50 --ckpt-freq 5 --output exp_batchnorm

# Experiment A2: ResNet-style
python train_cls.py ./configs/cls_miniplaces_cnn.yaml --print-freq 50 --ckpt-freq 5 --output exp_resnet

# Experiment A3: Inception-style
python train_cls.py ./configs/cls_miniplaces_cnn.yaml --print-freq 50 --ckpt-freq 5 --output exp_inception

# Experiment A4: Stochastic Depth
python train_cls.py ./configs/cls_miniplaces_cnn.yaml --print-freq 50 --ckpt-freq 5 --output exp_stochastic_depth

# Monitor training with TensorBoard:
tensorboard --logdir=../logs
""")

---

# SECTION G: RECOMMENDED EXPERIMENT PLAN

Based on lecture analysis and typical best practices, here's a recommended order:

---

In [ ]:
print("""
RECOMMENDED EXPERIMENT PLAN FOR A100 GPU (~8-15 min per experiment)
===================================================================

PHASE 1: QUICK ARCHITECTURE TESTS (30-45 min total)
---------------------------------------------------
1. Baseline SimpleNet (verify setup works)
2. SimpleNet + BatchNorm (A1) 
   - EXPECTED: +2-5% accuracy, faster convergence
   - This is the most impactful single change

PHASE 2: ARCHITECTURE IMPROVEMENTS (45-60 min total)
----------------------------------------------------
3. SimpleResNet with skip connections (A2)
   - EXPECTED: +3-5% over BatchNorm baseline
4. SimpleInceptionNet with parallel branches (A3)
   - EXPECTED: comparable to ResNet, different failure modes

PHASE 3: REGULARIZATION TUNING (30-45 min total)
------------------------------------------------
5. Best architecture + Stochastic Depth (A4)
   - EXPECTED: +1-2% via regularization
6. Best architecture + stronger weight decay (D1)
   - EXPECTED: improved generalization

PHASE 4: OPTIMIZER/SCHEDULE TUNING (30-45 min total)
----------------------------------------------------
7. Extended warmup + higher LR (C3)
   - EXPECTED: faster convergence to same accuracy
8. Step decay comparison (C2)
   - Good for understanding learning dynamics

PHASE 5: DATA AUGMENTATION (30-45 min total)
--------------------------------------------
9. Stronger augmentation (E1)
   - EXPECTED: +1-2% via regularization
10. Mixup (E2) - if time permits
    - EXPECTED: +1-2% additional

FINAL: COMBINED BEST SETTINGS
-----------------------------
11. Combine best architecture + best training settings
    - Should achieve best overall accuracy

ESTIMATED TOTAL TIME: 3-4 hours on A100
TARGET IMPROVEMENT: +10-15% over baseline SimpleNet
""")

---

# SECTION H: IMPLEMENTATION - COPY THIS TO modeling.py

Complete implementation code to add to `cls_libs/modeling.py`

---

In [ ]:
COMPLETE_MODELING_CODE = '''
################################################################################
# Part I.2: Design your own CNN - IMPROVED ARCHITECTURES
# Add this code AFTER the SimpleNet class in modeling.py
################################################################################

# Import DropPath from blocks.py (already imported at top of file)
# from .blocks import DropPath


class SimpleNetBN(nn.Module):
    """
    SimpleNet with Batch Normalization.
    Most impactful single improvement over baseline.
    """
    def __init__(self, conv_op=nn.Conv2d, num_classes=100):
        super(SimpleNetBN, self).__init__()
        
        self.features = nn.Sequential(
            conv_op(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            
            conv_op(64, 64, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            conv_op(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            conv_op(64, 256, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            
            conv_op(256, 128, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            conv_op(128, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            conv_op(128, 512, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


class ResidualBlock(nn.Module):
    """Basic residual block with skip connection."""
    def __init__(self, in_channels, out_channels, stride=1, drop_path_rate=0.0):
        super(ResidualBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0 else nn.Identity()
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.drop_path(out)
        out = self.relu(out + identity)
        return out


class ImprovedCNN(nn.Module):
    """
    Improved CNN combining best practices:
    - Batch Normalization
    - Skip Connections (ResNet-style)
    - Stochastic Depth
    - Proper initialization
    """
    def __init__(self, num_classes=100, drop_path_rate=0.1):
        super(ImprovedCNN, self).__init__()
        
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        
        # Linear drop path rate schedule
        total_blocks = 8
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, total_blocks)]
        
        self.layer1 = self._make_layer(64, 64, 2, stride=1, drop_rates=dpr[0:2])
        self.layer2 = self._make_layer(64, 128, 2, stride=2, drop_rates=dpr[2:4])
        self.layer3 = self._make_layer(128, 256, 2, stride=2, drop_rates=dpr[4:6])
        self.layer4 = self._make_layer(256, 512, 2, stride=2, drop_rates=dpr[6:8])
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(512, num_classes)
        
        self._initialize_weights()

    def _make_layer(self, in_ch, out_ch, num_blocks, stride, drop_rates):
        layers = [ResidualBlock(in_ch, out_ch, stride, drop_rates[0])]
        for i in range(1, num_blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1, drop_rates[i]))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


# CHANGE THIS TO YOUR MODEL!
# Options: SimpleNet, SimpleNetBN, ImprovedCNN
default_cnn_model = ImprovedCNN
'''

print("Complete Implementation Code for modeling.py")
print("="*60)
print(COMPLETE_MODELING_CODE)

---

# SUMMARY: KEY IMPROVEMENTS FROM LECTURE

| Technique | Impact | Implementation Difficulty |
|-----------|--------|---------------------------|
| Batch Normalization | HIGH (+2-5%) | Easy (add BN layers) |
| Skip Connections | HIGH (+3-5%) | Medium (restructure) |
| Stochastic Depth | MEDIUM (+1-2%) | Easy (use DropPath) |
| Weight Decay Tuning | MEDIUM (+1-2%) | Easy (YAML config) |
| Learning Rate Schedule | MEDIUM | Easy (YAML config) |
| Data Augmentation | MEDIUM (+1-3%) | Easy (YAML config) |
| Dropout | LOW-MEDIUM | Easy (add layers) |
| Label Smoothing | LOW (+0.5-1%) | Medium (modify loss) |
| Mixup | LOW-MEDIUM (+1-2%) | Medium (modify training) |

**Recommended Priority:**
1. Add BatchNorm → biggest single impact
2. Add skip connections → enables deeper networks
3. Tune weight decay and augmentation → easy wins
4. Add stochastic depth → regularization for deeper nets

---